![iceberg-logo](https://www.apache.org/logos/res/iceberg/iceberg.png)

# Icebergの仕組みを理解しよう

## 本章のねらい：
- **Icebergの基本的なアーキテクチャを理解する**  
- **実際のメタデータとデータの動きを手を動かしながら観察して理解を深める**  

## Introduction

本章では、Icebergを構成するメタデータとデータの内部構造を理解することを目指します。  
これらを理解することで、Icebergの高速性や、Time TravelやSchema Evolution、同時アクセス制御などが何故/どのように実現されているのかが分かるようになります。

### Icebergの本質はTable仕様である

最初に理解していただきたいのは、Icebergの本質はストレージエンジンでもクエリエンジンでもなく、**Table仕様**であるということです。  
Iceberg自体はテーブルフォーマットの仕様に過ぎず、それ自体が特定のソフトウェアやプロセスを指すものではありません。  

つまり、実際にIceberg Tableを扱うのはSparkやTrino, Hiveなどのエンジン（本ハンズオンではPyIceberg）であって、各エンジンがIcebergのTable仕様の約束に従ってデータ/メタデータを書き込み/読み取りすることでIcebergの機能が実現されるということです。

### Icebergのアーキテクチャ

Icebergは大きく3つのレイヤーで構成され、木構造になっています。  
([Iceberg Table Spec](https://iceberg.apache.org/spec/)に掲載されている図をぜひ確認してください)

- **カタログ (Catalog)**
- **メタデータ層 (Metadata Layer)**
- **データ層 (Data Layer)**

#### Iceberg Catalog

現在の最新断面のメタデータが格納されているmetadata fileのロケーションをポイントします。  
Reader, WriterはIceberg Catalogを参照することでテーブルの最新の状態を把握して、それを起点にmetadata fileの木構造を下っていきます。  

本ハンズオンでは **SQLiteカタログ** を使用します。SQLiteファイル1つで完結し、外部サービスは不要です。  
本番環境では AWS Glue, Hive Metastore, REST Catalogなどが使われます。

#### Metadata Layer

Metadata Layerはmetadata files、manifest lists、manifest filesの3階層に分かれます。  
これらはテーブルに対する全てのオペレーションを追跡しており、Timetravel クエリやSchema Evolutionなどが実現されます。

- **metadata file** (`*.metadata.json`): テーブルのスキーマ、パーティション情報、スナップショット一覧を保持するJSONファイル
- **manifest list** (`snap-*.avro`): あるスナップショットが参照するmanifest fileのリストを保持するAvroファイル
- **manifest file** (`*.avro`): data fileのパス、レコード数、統計情報(最小値/最大値など)を保持するAvroファイル

#### Data Layer

data filesはテーブルのレコードデータ本体で、Apache Parquet, Apache ORC, Apache Avroなどの形式がサポートされています。  
本ハンズオンではデフォルトの**Parquet形式**が使われます。

## ハンズオン

ここからは実際に手を動かしながら、ここまでに紹介したIcebergのアーキテクチャを体感していきます。

### 前準備

In [ ]:
import shutil, os, json, glob
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from datetime import datetime
from pyiceberg.catalog.sql import SqlCatalog
from pyiceberg.schema import Schema
from pyiceberg.types import (
    NestedField, LongType, TimestampType, DoubleType, StringType
)

WAREHOUSE_PATH = "/home/iceberg/warehouse"

# ノートブックを何度も再実行できるようにするため、既存のwarehouseを削除
shutil.rmtree(WAREHOUSE_PATH, ignore_errors=True)

catalog = SqlCatalog(
    "demo",
    **{
        "uri": f"sqlite:///{WAREHOUSE_PATH}/catalog.db",
        "warehouse": f"file://{WAREHOUSE_PATH}",
    },
)
catalog.create_namespace("nyc")
print("セットアップ完了")

In [ ]:
# ファイル構成を表示するヘルパー関数
def list_warehouse(subpath="nyc.db/taxis"):
    target = f"{WAREHOUSE_PATH}/{subpath}"
    if not os.path.exists(target):
        print(f"{target} は存在しません")
        return
    for root, dirs, files in os.walk(target):
        dirs.sort()
        level = root.replace(target, '').count(os.sep)
        indent = '  ' * level
        print(f"{indent}{os.path.basename(root)}/")
        for f in sorted(files):
            size = os.path.getsize(os.path.join(root, f))
            print(f"{indent}  {f}  ({size:,} bytes)")

### CREATE TABLEしてみる

ニューヨーク市タクシーデータのスキーマでIcebergテーブルを作成します。  
この時点ではスキーマ定義のみで、実際のデータは格納されません。

In [ ]:
schema = Schema(
    NestedField(1,  "vendor_id",            LongType(),      required=False),
    NestedField(2,  "pickup_datetime",       TimestampType(), required=False),
    NestedField(3,  "dropoff_datetime",      TimestampType(), required=False),
    NestedField(4,  "passenger_count",       DoubleType(),    required=False),
    NestedField(5,  "trip_distance",         DoubleType(),    required=False),
    NestedField(6,  "rate_code_id",          DoubleType(),    required=False),
    NestedField(7,  "store_and_fwd_flag",    StringType(),    required=False),
    NestedField(8,  "pu_location_id",        LongType(),      required=False),
    NestedField(9,  "do_location_id",        LongType(),      required=False),
    NestedField(10, "payment_type",          LongType(),      required=False),
    NestedField(11, "fare_amount",           DoubleType(),    required=False),
    NestedField(12, "extra",                 DoubleType(),    required=False),
    NestedField(13, "mta_tax",               DoubleType(),    required=False),
    NestedField(14, "tip_amount",            DoubleType(),    required=False),
    NestedField(15, "tolls_amount",          DoubleType(),    required=False),
    NestedField(16, "improvement_surcharge", DoubleType(),    required=False),
    NestedField(17, "total_amount",          DoubleType(),    required=False),
    NestedField(18, "congestion_surcharge",  DoubleType(),    required=False),
    NestedField(19, "airport_fee",           DoubleType(),    required=False),
)

table = catalog.create_table("nyc.taxis", schema=schema)
print(f"テーブル作成完了: {table.name()}")

`nyc/taxis/metadata/` を見ると、`*.metadata.json` が作られています。これが **metadata file** です。

In [ ]:
print("=== CREATE TABLE 直後のファイル構成 ===")
list_warehouse()

metadata fileの中身を見ると、カラムやデータ型などのスキーマ情報が格納されています。  
今の時点ではスキーマを定義したのみで実際のデータは格納されていないため、`snapshots` は空になっています。

In [ ]:
meta_files = sorted(glob.glob(f"{WAREHOUSE_PATH}/nyc.db/taxis/metadata/*.metadata.json"))
print(f"metadata file: {os.path.basename(meta_files[0])}")
print()

with open(meta_files[0]) as f:
    meta0 = json.load(f)

# スキーマとスナップショットの部分だけ確認
print("--- スキーマ情報 ---")
print(json.dumps(meta0.get("schemas", meta0.get("schema", {})), indent=2, ensure_ascii=False))
print()
print("--- スナップショット一覧 ---")
print(json.dumps(meta0.get("snapshots", []), indent=2))

### INSERTしてみる

3件のレコードをINSERTします。  
INSERTを実行すると、metadata layerにどのようなファイルが追加されるかを観察しましょう。

In [ ]:
rows = [
    (1, datetime(2023,3,1,8,0,0),  datetime(2023,3,1,8,15,0),  1.0, 3.5, 1.0, 'N', 263, 161, 1, 12.5, 0.5, 0.5, 2.0, 0.0, 0.3, 15.8, 0.0, 0.0),
    (2, datetime(2023,3,1,9,0,0),  datetime(2023,3,1,9,20,0),  2.0, 2.8, 1.0, 'N', 186, 230, 2, 10.0, 0.5, 0.5, 0.0, 0.0, 0.3, 11.3, 2.5, 0.0),
    (1, datetime(2023,3,1,10,0,0), datetime(2023,3,1,10,35,0), 1.0, 5.0, 1.0, 'N', 162, 234, 1, 18.0, 0.5, 0.5, 3.0, 0.0, 0.3, 22.3, 0.0, 0.0),
]
df = pd.DataFrame(rows, columns=[f.name for f in table.schema().fields])
arrow_table = pa.Table.from_pandas(df, schema=table.schema().as_arrow())
table.append(arrow_table)

print(f"INSERT完了: {len(table.scan().to_arrow())} レコード")

改めてファイル構成を見てみると、3種類の新しいファイルが作られています。

- 新しい `*.metadata.json` (metadata file)
- `snap-*.avro` (manifest list)
- `*.avro` (manifest file)
- `data/*.parquet` (data file)

In [ ]:
print("=== INSERT後のファイル構成 ===")
list_warehouse()

#### metadata fileの変化

新しい `*.metadata.json` と比較すると、`snapshots` にINSERTのスナップショットが記録されているのが分かります。

In [ ]:
meta_files = sorted(glob.glob(f"{WAREHOUSE_PATH}/nyc.db/taxis/metadata/*.metadata.json"))
print(f"最新 metadata file: {os.path.basename(meta_files[-1])}")
print()

with open(meta_files[-1]) as f:
    meta1 = json.load(f)

print("--- last-sequence-number ---")
print(meta1.get("last-sequence-number"))
print()
print("--- current-snapshot-id ---")
print(meta1.get("current-snapshot-id"))
print()
print("--- snapshots ---")
for s in meta1.get("snapshots", []):
    print(json.dumps(s, indent=2))

#### manifest list (snap-\*.avro) の内容

PyIcebergのAPIを使って、manifest listが保持する情報を確認します。  
manifest listはあるスナップショットに紐づくmanifest fileのリスト（パスや統計情報）を保持しています。

In [ ]:
snap = table.current_snapshot()
print(f"snapshot_id     : {snap.snapshot_id}")
print(f"timestamp_ms    : {snap.timestamp_ms}")
print(f"operation       : {snap.summary.operation.value if snap.summary else 'unknown'}")
print(f"manifest_list   : {os.path.basename(snap.manifest_list)}")
print()

print("--- manifest list が保持する manifest files ---")
manifests = snap.manifests(table.io)
rows = []
for m in manifests:
    rows.append({
        "manifest_file": os.path.basename(m.manifest_path),
        "added_files_count": m.added_files_count,
        "existing_files_count": m.existing_files_count,
        "deleted_files_count": m.deleted_files_count,
    })
pd.DataFrame(rows)

#### manifest file (\*.avro) の内容

manifest fileは、data fileのパス、レコード数、ファイルサイズ、各カラムの統計情報（最小値/最大値）を保持しています。  
クエリエンジンはこれらの統計情報を元に、読み込み対象のファイルをプルーニング（最適化）します。

In [ ]:
print("--- manifest file が保持する data files の情報 ---")
file_rows = []
for task in table.scan().plan_files():
    df_file = task.file
    file_rows.append({
        "file_path": os.path.basename(df_file.file_path),
        "file_format": str(df_file.file_format),
        "record_count": df_file.record_count,
        "file_size_in_bytes": df_file.file_size_in_bytes,
    })
pd.DataFrame(file_rows)

#### data file (\*.parquet) の内容

data fileはParquet形式で保存されたテーブルの実データです。  
PyArrowで直接読み込んで確認してみましょう。

In [ ]:
data_files = sorted(glob.glob(f"{WAREHOUSE_PATH}/nyc.db/taxis/data/**/*.parquet", recursive=True))
print(f"data file: {os.path.basename(data_files[0])}")
print()
pq.read_table(data_files[0]).to_pandas()

### もう一度INSERTしてみる

もう一度INSERTを実行してみます。  
スナップショットが蓄積されていく様子を観察しましょう。

In [ ]:
rows2 = [
    (1, datetime(2023,3,2,8,0,0),  datetime(2023,3,2,8,20,0),  3.0, 4.2, 1.0, 'N', 100, 200, 1, 15.0, 0.5, 0.5, 2.5, 0.0, 0.3, 18.8, 0.0, 0.0),
    (2, datetime(2023,3,2,9,0,0),  datetime(2023,3,2,9,15,0),  1.0, 1.8, 1.0, 'N',  50, 120, 2,  8.0, 0.5, 0.5, 0.0, 0.0, 0.3,  9.3, 0.0, 0.0),
]
df2 = pd.DataFrame(rows2, columns=[f.name for f in table.schema().fields])
table.append(pa.Table.from_pandas(df2, schema=table.schema().as_arrow()))

print(f"2回目のINSERT完了: 合計 {len(table.scan().to_arrow())} レコード")

In [ ]:
print("=== 2回目のINSERT後のファイル構成 ===")
list_warehouse()

INSERTのたびに `*.metadata.json`、`snap-*.avro`、`*.avro`、`*.parquet` が追加されているのが分かります。  
このように、Icebergはデータの変更を全てスナップショットとして追記保存します（上書きしません）。これがタイムトラベルの仕組みの基盤となっています。

In [ ]:
# スナップショットの蓄積を確認
meta_files = sorted(glob.glob(f"{WAREHOUSE_PATH}/nyc.db/taxis/metadata/*.metadata.json"))
with open(meta_files[-1]) as f:
    meta_latest = json.load(f)

print(f"スナップショット数: {len(meta_latest.get('snapshots', []))}")
print()

rows_snap = []
for s in meta_latest.get("snapshots", []):
    rows_snap.append({
        "snapshot-id": s.get("snapshot-id"),
        "timestamp-ms": s.get("timestamp-ms"),
        "operation": s.get("summary", {}).get("operation"),
        "added-records": s.get("summary", {}).get("added-records"),
        "total-records": s.get("summary", {}).get("total-records"),
    })
pd.DataFrame(rows_snap)

### まとめ

本章では、Icebergテーブルの内部構造を実際のファイルを観察しながら確認しました。

| ファイル種別 | 形式 | 役割 |
|---|---|---|
| `*.metadata.json` | JSON | スキーマ・スナップショット一覧の管理 |
| `snap-*.avro` | Avro | manifest fileのリスト（manifest list） |
| `*.avro` | Avro | data fileのパス・統計情報（manifest file） |
| `data/*.parquet` | Parquet | テーブルの実データ（data file） |

**重要なポイント：**
- データの変更（INSERT/UPDATE/DELETE）が発生するたびに、新しいスナップショットとして追記保存される
- クエリエンジンはカタログ → metadata.json → manifest list → manifest file → data file の順に木構造を辿ってデータを読む
- manifest fileの統計情報（最小値/最大値）によりクエリ対象ファイルをプルーニングして高速化する
- このアーキテクチャがTime Travel、Schema Evolution、同時実行制御の基盤となっている